In [1]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from default_risk.scripts.auxiliars_for_modeling import get_pipeline

import logging
from contextlib import redirect_stderr, redirect_stdout
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria
from default_risk.scripts.auxiliars_for_modeling import apply_cyclical_encoding

logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("mlflow.tracking._tracking_service.client").setLevel(logging.ERROR)

# Mostrar TODAS las filas del DataFrame
pd.set_option('display.max_rows', None)

# Mostrar TODAS las columnas (crucial para tus 360+ features)
pd.set_option('display.max_columns', None)



# Ajustar el ancho de la pantalla para que no se rompa la tabla en la consola
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)





In [7]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)


model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)

cols_to_drop= ["name_income_type"]



importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)

X= X.drop(columns=cols_to_drop)

last= pd.read_csv(cfg.ARTIFACTS_DIR / "plain_filter.csv")

X= clean_noise_from_feature_importance(last,X,0.002)

#five_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_fine_pruned_final_model.csv")

#X = clean_importance_zero_and_negative_pfi(five_filter,X,0.00009)



eliminando ['diff_application_credit_median', 'building_score_mean', 'log_amt_application_std', 'closed_days_credit_update_closed_mean', 'credit_card_is_over_the_limit_sum_prev_1', 'active_days_credit_active_min', 'ratio_credit_to_annuity_prev_1', 'bureau_balance_amount_rows_with_activity_loan_1', 'closed_balance_months_balance_min_closed_min', 'instalments_repeated_for_underpayment_mean_prev_1', 'log_amt_credit_std', 'credit_card_amt_balance_std_prev_1', 'global_approval_ratio', 'log_amt_credit_mean', 'fondkapremont_mode', 'weekday_appr_process_start_sin', 'flag_live_city_not_work', 'active_amt_annuity_active_mean', 'instalments_log_amt_instalment_mean_prev_1', 'active_amt_annuity_active_std', 'amt_req_credit_breau_mon', 'instalments_raw_size_serie_prev_1', 'active_balance_months_balance_min_active_min', 'instalments_is_delinquency_sum_prev_1', 'building_score_std', 'log_diff_application_credit_prev_1', 'nflag_insured_on_approval_sum', 'active_amt_credit_sum_limit_active_std', 'bureau

In [ ]:
columns = X.columns
columns= columns.drop( ["organization_type","occupation_type"])
#indice_inicio = X.columns.get_loc("amt_down_payment_sum")
#columnas_restantes = X.columns[indice_inicio:]

model= pipeline
baseline_oof_auc, baseline_std = run_cv_tracked_mlflow(model,hiperparams,cv,X,Y,experiment_name,"best-features",save_final_model=False,persist_feature_importance=False)


results = []
features_drop_file = cfg.ARTIFACTS_DIR / 'leave_one_out.csv'
pd.DataFrame(columns=['feature_dropped', 'auc_impact', 'std_impact']).to_csv(features_drop_file, index=False, encoding='utf-8')


from tqdm.auto import tqdm
for col in tqdm(columns, desc="Evaluating model without variables"):
        X_dropped = X.drop(columns=[col])
        run_name = f"best-features_{col.replace('/', '_')}"
        oof_auc, std = run_cv_tracked_mlflow(clone(model), hiperparams, cv, X_dropped, Y, experiment_name, run_name=run_name,save_final_model=False,persist_feature_importance=False)
        auc_drop = baseline_oof_auc - oof_auc
        std_diff = baseline_std - std
        results.append({
                    'feature_dropped': col,
                    'auc_impact': auc_drop,
                    'std_impact': std_diff
                })
        
        row_df = pd.DataFrame([{
        'feature_dropped': col,
        'auc_impact': auc_drop,
        'std_impact': std_diff
    }])
    
        row_df.to_csv(features_drop_file, mode='a', header=False, index=False, encoding='utf-8')
        print(f"Feature {col} dropped. Result: {auc_drop}, {std_diff}")



del merged_df
gc.collect()

🏃 View run best-features_child_1 at: http://localhost:5000/#/experiments/3/runs/854a26699e2d48748841ffcb6695fa97
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_2 at: http://localhost:5000/#/experiments/3/runs/ca563f82ce7645a294ff0afedd71658d
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_3 at: http://localhost:5000/#/experiments/3/runs/df598e0ad10049f7bae8714a99270177
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_4 at: http://localhost:5000/#/experiments/3/runs/ff96c8c498bb4e89896864c0c30b2d30
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_child_5 at: http://localhost:5000/#/experiments/3/runs/9e72d171e79b41298d78430014f8c249
🧪 View experiment at: http://localhost:5000/#/experiments/3
AUC per fold= 0.786 ± 0.004(std), auc_score_OOF= 0.786 result of CV with 5 folds. 
🏃 View run Parent_best-features at: http://localhost

Evaluating model without variables:   0%|          | 0/252 [00:00<?, ?it/s]

🏃 View run best-features_code_gender_child_1 at: http://localhost:5000/#/experiments/3/runs/d61eab32891b4426b2d9349e4291b325
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_2 at: http://localhost:5000/#/experiments/3/runs/c2a98a5a28a0413da63d20c485c5e005
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_3 at: http://localhost:5000/#/experiments/3/runs/b0976a5e588744cdb7cbd3b369b15255
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_4 at: http://localhost:5000/#/experiments/3/runs/b105791ea26d4c689638c86cf58e7269
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_code_gender_child_5 at: http://localhost:5000/#/experiments/3/runs/1d1030c8310841ff9352117b0ecc03e6
🧪 View experiment at: http://localhost:5000/#/experiments/3
AUC per fold= 0.782 ± 0.003(std), auc_score_OOF= 0.782 result of CV with 5 

Evaluating model without variables:   0%|          | 1/252 [02:29<10:23:23, 149.02s/it]

Feature code_gender dropped. Result: 0.003619075138228789, 0.0004914124406304333
🏃 View run best-features_cnt_children_child_1 at: http://localhost:5000/#/experiments/3/runs/b7436ea1bc894a53a0bef79f28125bdf
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_cnt_children_child_2 at: http://localhost:5000/#/experiments/3/runs/0f5a8f4b4601437f9cbf5363b2e6f62a
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_cnt_children_child_3 at: http://localhost:5000/#/experiments/3/runs/8e340df158c7495684d1cda8efe1b9e0
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_cnt_children_child_4 at: http://localhost:5000/#/experiments/3/runs/4c51504ca9f041898f8d8c77cf1de769
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_cnt_children_child_5 at: http://localhost:5000/#/experiments/3/runs/214e6ccb43404423bcf19ada32ec7077
🧪 View experiment at: http://localhost:5000/#/exp

...

🏃 View run Parent_best-features_bureau_ratio_debt_limit_loan_1 at: http://localhost:5000/#/experiments/3/runs/34135671da374f9a99ec6b667b71fdd4
🧪 View experiment at: http://localhost:5000/#/experiments/3
Feature bureau_ratio_debt_limit_loan_1 dropped. Result: 0.0011853822159030303, 0.00069121916913676
🏃 View run best-features_active_id_curr_active_count_child_1 at: http://localhost:5000/#/experiments/3/runs/148dc652140a4837b7f6fb1539264d77
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_active_id_curr_active_count_child_2 at: http://localhost:5000/#/experiments/3/runs/aca98bbb6f664f65ba4ff846b9198c29
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_active_id_curr_active_count_child_3 at: http://localhost:5000/#/experiments/3/runs/f2857f32911a42b3a1c2b5b642749978
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_active_id_curr_active_count_child_4 at: http://localhost:5000/#/experi

Evaluating model without variables:  26%|██▌       | 66/252 [10:41:59<9:16:59, 179.68s/it]

AUC per fold= 0.784 ± 0.003(std), auc_score_OOF= 0.784 result of CV with 5 folds. 
🏃 View run Parent_best-features_active_id_curr_active_count at: http://localhost:5000/#/experiments/3/runs/75cae187bf704b55a9dd817db033ac7b
🧪 View experiment at: http://localhost:5000/#/experiments/3
Feature active_id_curr_active_count dropped. Result: 0.0014064383513558987, 0.001005654753750019
🏃 View run best-features_active_amt_credit_sum_active_max_child_1 at: http://localhost:5000/#/experiments/3/runs/48ba82d0554448148ab576e7e0cacd7e
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_active_amt_credit_sum_active_max_child_2 at: http://localhost:5000/#/experiments/3/runs/167693770d7a489e98d6b51822dbb529
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_active_amt_credit_sum_active_max_child_3 at: http://localhost:5000/#/experiments/3/runs/1e91bac06e044405a9d49e7a5903d74b
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 

Evaluating model without variables:  27%|██▋       | 67/252 [10:44:37<8:53:53, 173.15s/it]

Feature active_amt_credit_sum_active_max dropped. Result: 0.0016971796063163236, 0.0012983878091237981
🏃 View run best-features_active_amt_credit_sum_active_mean_child_1 at: http://localhost:5000/#/experiments/3/runs/9f5738d3d98e4808bb084ea8b58d6196
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run best-features_active_amt_credit_sum_active_mean_child_2 at: http://localhost:5000/#/experiments/3/runs/8c66d6a7f75849e1a8213febf372c648
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [3]:
print("Preparando dataset temporal...")
df_temp = X.copy()
umbral = 0.85
# 1. Transformar texto a números y manejar nulos
for col in df_temp.columns:
    if df_temp[col].dtype == 'object' or df_temp[col].dtype.name == 'category':
        # pd.factorize asigna un número único a cada categoría de texto.
        # Automáticamente asigna el valor -1 a los nulos (NaNs).
        # Esto es perfecto para XGBoost porque agrupa los nulos en una sola rama.
        df_temp[col] = pd.factorize(df_temp[col])[0]

print("Calculando matriz de correlación (Spearman)...")
# 2. Calcular matriz Spearman (ideal porque captura relaciones de orden no lineales)
matriz_corr = df_temp.corr(method='spearman').abs()

# 3. Tomar solo el triángulo superior para evitar A-B y B-A
triangulo_superior = matriz_corr.where(
    np.triu(np.ones(matriz_corr.shape), k=1).astype(bool)
)

# 4. Encontrar los pares problemáticos
print(f"Buscando pares con correlación mayor a {umbral}...")
pares_redundantes = []
variables_a_revisar = set()

for col in triangulo_superior.columns:
    alta_corr = triangulo_superior.index[triangulo_superior[col] > umbral].tolist()
    for row in alta_corr:
        correlacion_valor = round(triangulo_superior.loc[row, col], 3)
        pares_redundantes.append((row, col, correlacion_valor))
        variables_a_revisar.add(row)
        variables_a_revisar.add(col)
        
# Ordenar los resultados de mayor a menor correlación
pares_redundantes.sort(key=lambda x: x[2], reverse=True)


lista_sospechosas = list(variables_a_revisar)

# --- CÓMO USARLO EN TU CÓDIGO ---
# X_train es tu dataset original de 500 variables (con sus textos y nulos intactos)

pares = pares_redundantes

print(f"\nSe encontraron {len(pares)} pares de variables altamente redundantes.")
print("Top 5 pares más correlacionados:")
for p in pares[:5]:
    print(f" - {p[0]} <---> {p[1]} (Corr: {p[2]})")
    

Preparando dataset temporal...
Calculando matriz de correlación (Spearman)...
Buscando pares con correlación mayor a 0.85...

Se encontraron 293 pares de variables altamente redundantes.
Top 5 pares más correlacionados:
 - bureau_days_credit_loan_2 <---> bureau_balance_months_balance_min_loan_2 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_1 <---> bureau_balance_status_score_std_loan_1 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_2 <---> bureau_balance_status_score_std_loan_2 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_1 <---> bureau_balance_is_delincuency_mean_loan_1 (Corr: 1.0)
 - bureau_balance_status_score_mean_loan_2 <---> bureau_balance_is_delincuency_mean_loan_2 (Corr: 1.0)


In [4]:
df = pd.read_csv(cfg.ARTIFACTS_DIR / 'long-road-2_feature_importance.csv')
zero_imp_features = df[df['importances'] == 0.0]['feature_name'].tolist()
print(f"Total zero importance features: {len(zero_imp_features)}")
print(zero_imp_features)

Total zero importance features: 82
['closed_amt_credit_sum_limit_closed_max', 'bureau_have_amt_credit_sum_overdue_loan_1', 'bureau_amt_credit_sum_limit_short_limit_loan_2', 'bureau_amt_credit_sum_limit_short_limit_loan_1', 'bureau_amt_credit_sum_limit_is_missing_loan_2', 'bureau_amt_credit_sum_limit_is_missing_loan_1', 'bureau_amt_credit_max_overdue_is_missing_loan_2', 'bureau_days_enddate_fact_is_missing_loan_2', 'days_first_drawing_has_sentinel_value_prev_1', 'bureau_days_credit_enddate_third_positive_cluster_loan_1', 'bureau_days_credit_enddate_first_positive_cluster_loan_1', 'bureau_days_credit_enddate_closed_loan_2', 'bureau_days_credit_enddate_closed_loan_1', 'bureau_have_amt_credit_sum_overdue_loan_2', 'bureau_flag_have_credit_day_overdue_loan_2', 'days_first_due_has_sentinel_value_prev_1', 'organization_type_University', 'days_termination_has_sentinel_value_prev_1', 'organization_type_Trade: type 7', 'organization_type_Trade: type 3', 'organization_type_Medicine', 'organization

In [16]:


def purga_inteligente(lista_correlaciones, lista_cero_impacto):
    from collections import defaultdict
    
    # Construimos un grafo para agrupar todas las variables que se relacionan en cadena
    grafo = defaultdict(list)
    for u, v, _ in lista_correlaciones:
        grafo[u].append(v)
        grafo[v].append(u)
        
    visitados = set()
    grupos_correlacionados = []
    
    # Encontrar todos los grupos de variables que comparten información
    for nodo in grafo:
        if nodo not in visitados:
            grupo = set()
            cola = [nodo]
            while cola:
                actual = cola.pop(0)
                if actual not in visitados:
                    visitados.add(actual)
                    grupo.add(actual)
                    cola.extend(grafo[actual])
            grupos_correlacionados.append(grupo)

    set_cero_impacto = set(lista_cero_impacto)
    variables_correlacionadas = set(grafo.keys())
    
    # Listas de resultados
    borrar_basura_pura = []
    borrar_cubiertas = []
    salvar_representantes = []
    
    # 1. Variables que no están correlacionadas con nada (Basura pura)
    for var in set_cero_impacto:
        if var not in variables_correlacionadas:
            borrar_basura_pura.append(var)
            
    # 2. Analizar los grupos correlacionados
    for grupo in grupos_correlacionados:
        # Variables de este grupo que ibas a borrar
        variables_a_borrar_aqui = grupo.intersection(set_cero_impacto)
        # Variables de este grupo que son BUENAS (no están en tu lista de borrar)
        variables_buenas_aqui = grupo - variables_a_borrar_aqui
        
        if not variables_a_borrar_aqui:
            continue # Ninguna variable de este grupo iba a ser borrada, lo ignoramos
            
        if len(variables_buenas_aqui) > 0:
            # Hay al menos una variable buena que guarda esta información.
            # Podemos borrar todas las variables de impacto 0 de este grupo.
            borrar_cubiertas.extend(list(variables_a_borrar_aqui))
        else:
            # PELIGRO: Todas las variables de este grupo están en tu lista de borrar.
            # Se enmascararon mutuamente. Debemos salvar a la primera y borrar el resto.
            lista_peligro = list(variables_a_borrar_aqui)
            salvada = lista_peligro[0]
            borradas = lista_peligro[1:]
            
            salvar_representantes.append(salvada)
            borrar_cubiertas.extend(borradas)

    return borrar_basura_pura, borrar_cubiertas, salvar_representantes

# Ejecutamos la función

lista_correlaciones_estricta = [
    (var1, var2, corr) for var1, var2, corr in pares if corr >= 0.999
]

basura_pura, cubiertas, salvadas = purga_inteligente(lista_correlaciones_estricta, zero_imp_features)

# --- IMPRESIÓN DE RESULTADOS ---
print("--- RESULTADOS DE LA PURGA INTELIGENTE ---\n")

print(f"✅ 1. VARIABLES SALVADAS (Efecto Sombra detectado): {len(salvadas)}")
print("Conserva estas variables en tu dataset. Se anularon entre sí, pero si las borras todas pierdes la información:")
for v in salvadas:
    print(f"  -> {v}")

print(f"\n🗑️ 2. BORRAR SIN MIEDO (Basura Pura - Sin correlación): {len(basura_pura)}")
print("No aportan nada ni encubren a nadie:")
# Imprime solo 5 como ejemplo para no saturar la pantalla
print(basura_pura[:5], "...\n") 

print(f"🗑️ 3. BORRAR SIN MIEDO (Redundantes cubiertas): {len(cubiertas)}")
print("Dieron 0 impacto y otra variable que YA conservas en el modelo se encarga de esa información:")
# Imprime solo 5 como ejemplo
print(cubiertas[:5], "...\n")

lista_final_a_borrar = basura_pura + cubiertas
print(f"\nRESUMEN: De las {len(inhert_features)} variables originales, borrarás {len(lista_final_a_borrar)} y salvarás {len(salvadas)}.")

--- RESULTADOS DE LA PURGA INTELIGENTE ---

✅ 1. VARIABLES SALVADAS (Efecto Sombra detectado): 1
Conserva estas variables en tu dataset. Se anularon entre sí, pero si las borras todas pierdes la información:
  -> active_cnt_credit_prolong_active_max

🗑️ 2. BORRAR SIN MIEDO (Basura Pura - Sin correlación): 74
No aportan nada ni encubren a nadie:
['flag_invalid_surface_sellerplace_area_prev_1', 'cnt_children', 'bureau_flag_have_credit_day_overdue_loan_1', 'closed_balance_months_balance_max_closed_max', 'bureau_has_bureau_balance_data_loan_2'] ...

🗑️ 3. BORRAR SIN MIEDO (Redundantes cubiertas): 7
Dieron 0 impacto y otra variable que YA conservas en el modelo se encarga de esa información:
['bureau_balance_status_score_mean_loan_1', 'bureau_balance_is_delincuency_mean_loan_1', 'bureau_balance_status_score_mean_loan_2', 'active_cnt_credit_prolong_active_mean', 'closed_amt_credit_sum_debt_closed_max'] ...


RESUMEN: De las 56 variables originales, borrarás 81 y salvarás 1.


In [17]:
print(cubiertas)

['bureau_balance_status_score_mean_loan_1', 'bureau_balance_is_delincuency_mean_loan_1', 'bureau_balance_status_score_mean_loan_2', 'active_cnt_credit_prolong_active_mean', 'closed_amt_credit_sum_debt_closed_max', 'amt_goods_price_sum', 'bureau_ratio_debt_limit_loan_1']
